In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Per-file summary: total row count, earliest timestamp (UTC), latest timestamp (UTC)
def file_summary(df, name):
    if df.shape[0] == 0:
        return pd.DataFrame([{
            'file': name,
            'rows': 0,
            'earliest_utc': None,
            'latest_utc': None
        }])
    return pd.DataFrame([{
        'file': name,
        'rows': df.shape[0],
        'earliest_utc': df['timestamp'].min().isoformat(),
        'latest_utc': df['timestamp'].max().isoformat()
    }])

summary_df = pd.concat([
    file_summary(metric_df, 'metric.csv'),
    file_summary(trace_df, 'trace.csv'),
    file_summary(log_df, 'log.csv'),
    file_summary(error_df, 'error_logs.csv')
], ignore_index=True)

# 2) metric.csv: unique (cmdb_id, kpi_name) pairs with row counts, sorted desc, top 20
if set(['cmdb_id','kpi_name']).issubset(metric_df.columns) and metric_df.shape[0] > 0:
    metric_pairs = (
        metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    metric_pairs = pd.DataFrame(columns=['cmdb_id','kpi_name','count'])

# 3) trace.csv: unique (cmdb_id, trace_name) pairs with row counts, sorted desc, top 20
if set(['cmdb_id','trace_name']).issubset(trace_df.columns) and trace_df.shape[0] > 0:
    trace_pairs = (
        trace_df.groupby(['cmdb_id','trace_name'], dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    trace_pairs = pd.DataFrame(columns=['cmdb_id','trace_name','count'])

# 4) log.csv: unique (cmdb_id, log_name) pairs with row counts, sorted desc, top 20
if set(['cmdb_id','log_name']).issubset(log_df.columns) and log_df.shape[0] > 0:
    log_pairs = (
        log_df.groupby(['cmdb_id','log_name'], dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    log_pairs = pd.DataFrame(columns=['cmdb_id','log_name','count'])

# 5) error_logs.csv: unique cmdb_id with counts and up to 5 sample rows (timestamp, cmdb_id, message)
if 'cmdb_id' in error_df.columns and error_df.shape[0] > 0:
    error_counts = (
        error_df.groupby('cmdb_id', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )
    # up to 5 sample rows across the file (ordered by timestamp)
    error_samples = error_df.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(5).reset_index(drop=True)
    # format sample timestamps as ISO
    error_samples['timestamp'] = error_samples['timestamp'].dt.isoformat()
else:
    error_counts = pd.DataFrame(columns=['cmdb_id','count'])
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Display compact results (variables - IPython will show them)
summary_df, metric_pairs, trace_pairs, log_pairs, error_counts, error_samples

```
Out[1]:
```
import pandas as pd

# Build a concise plain-English summary using existing variables from prior execution:
lines = []

# 1) Per-file summaries
for _, row in summary_df.iterrows():
    file = row['file']
    rows = int(row['rows'])
    earliest = row['earliest_utc'] if pd.notna(row['earliest_utc']) else "N/A"
    latest = row['latest_utc'] if pd.notna(row['latest_utc']) else "N/A"
    lines.append(f"{file}: {rows} rows, earliest={earliest}, latest={latest}")

lines.append("")  # blank line

# 2) Top metric (cmdb_id, kpi_name) pairs (top 5)
lines.append("Top metric (cmdb_id, kpi_name, count) — top 5:")
if not metric_pairs.empty:
    for _, r in metric_pairs.head(5).iterrows():
        lines.append(f"- {r['cmdb_id']} | {r['kpi_name']} | count={int(r['count'])}")
else:
    lines.append("- (no metric pairs)")

lines.append("")

# 3) Top trace (cmdb_id, trace_name) pairs (top 5)
lines.append("Top trace (cmdb_id, trace_name, count) — top 5:")
if not trace_pairs.empty:
    for _, r in trace_pairs.head(5).iterrows():
        lines.append(f"- {r['cmdb_id']} | {r['trace_name']} | count={int(r['count'])}")
else:
    lines.append("- (no trace pairs)")

lines.append("")

# 4) Top log (cmdb_id, log_name) pairs (top 5)
lines.append("Top log (cmdb_id, log_name, count) — top 5:")
if not log_pairs.empty:
    for _, r in log_pairs.head(5).iterrows():
        lines.append(f"- {r['cmdb_id']} | {r['log_name']} | count={int(r['count'])}")
else:
    lines.append("- (no log pairs)")

lines.append("")

# 5) error_logs summary
if error_counts.empty:
    lines.append("error_logs.csv: no rows found.")
else:
    lines.append("error_logs counts by cmdb_id (top):")
    for _, r in error_counts.head(5).iterrows():
        lines.append(f"- {r['cmdb_id']} | count={int(r['count'])}")
    lines.append("")
    lines.append("Up to 5 sample error log rows (timestamp, cmdb_id, message):")
    for _, r in error_samples.iterrows():
        ts = r['timestamp']
        cmdb = r['cmdb_id']
        msg = r['message']
        lines.append(f"- {ts} | {cmdb} | {msg}")

# Final summary string
summary = "\n".join(lines)

# Display the summary (IPython will show the variable)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows               earliest_utc                 latest_utc
0      metric.csv   8948  2024-01-24T18:58:00+00:00  2024-01-24T19:22:00+00:00
1       trace.csv  18640  2024-01-24T18:58:00+00:00  2024-01-24T19:22:00+00:00
2         log.csv   2200  2024-01-24T18:58:00+00:00  2024-01-24T19:22:00+00:00
3  error_logs.csv      0                       None                       None,                         cmdb_id    kpi_name  count
0            ts-voucher-service      socket     25
1   ts-admin-basic-info-service         cpu     25
2   ts-admin-basic-info-service      diskio     25
3   ts-admin-basic-info-service  latency-50     25
4   ts-admin-basic-info-service  latency-90     25
5   ts-admin-basic-info-service         mem     25
6              ts-travel2-mongo         mem     25
7              ts-travel2-mongo      socket     25
8            ts-travel2-service         cpu     25
9            ts-travel2-service      diskio     25
10           ts-travel2-service  latency-50     25
11           ts-travel2-service  latency-90     25
12           ts-travel2-service         mem     25
13           ts-travel2-service      socket     25
14            ts-travel-service      diskio     25
15            ts-travel-service  latency-50     25
16            ts-travel-service  latency-90     25
17            ts-travel-service         mem     25
18            ts-travel-service      socket     25
19            ts-travel-service    workload     25,                cmdb_id                                         trace_name  count
0      ts-user-service                 trace.to_ts-user-service.row_count     25
1                 root  trace.to_ts-admin-basic-info-service.duration_...     25
2                 root  trace.to_ts-admin-basic-info-service.duration_p95     25
3                 root    trace.to_ts-admin-basic-info-service.error_rate     25
4                 root     trace.to_ts-admin-basic-info-service.row_count     25
5                 root     trace.to_ts-admin-travel-service.duration_mean     25
6                 root      trace.to_ts-admin-travel-service.duration_p95     25
7                 root        trace.to_ts-admin-travel-service.error_rate     25
8                 root         trace.to_ts-admin-travel-service.row_count     25
9                 root        trace.to_ts-assurance-service.duration_mean     25
10                root         trace.to_ts-assurance-service.duration_p95     25
11                root           trace.to_ts-assurance-service.error_rate     25
12                root            trace.to_ts-assurance-service.row_count     25
13                root             trace.to_ts-auth-service.duration_mean     25
14                root              trace.to_ts-auth-service.duration_p95     25
15                root                trace.to_ts-auth-service.error_rate     25
16  ts-travel2-service      trace.to_ts-order-other-service.duration_mean     25
17  ts-travel2-service       trace.to_ts-order-other-service.duration_p95     25
18  ts-travel2-service         trace.to_ts-order-other-service.error_rate     25
19  ts-travel2-service          trace.to_ts-order-other-service.row_count     25,                         cmdb_id         log_name  count
0   ts-admin-basic-info-service  log.error_count     25
1   ts-admin-basic-info-service    log.row_count     25
2       ts-admin-travel-service  log.error_count     25
3       ts-admin-travel-service    log.row_count     25
4          ts-assurance-service    log.row_count     25
5          ts-assurance-service  log.error_count     25
6               ts-auth-service  log.error_count     25
7               ts-auth-service    log.row_count     25
8            ts-consign-service  log.error_count     25
9            ts-consign-service    log.row_count     25
10             ts-basic-service    log.row_count     25
11             ts-basic-service  log.error_count     25
12          ts-contacts-service  log.error_count     25
13          ts-contacts-service    log.row_count     25
14             ts-route-service    log.row_count     25
15             ts-route-service  log.error_count     25
16              ts-seat-service  log.error_count     25
17              ts-seat-service    log.row_count     25
18             ts-order-service  log.error_count     25
19             ts-order-service    log.row_count     25, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Load metric.csv if not already loaded in the kernel
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp parsed as UTC datetime
if not pd.api.types.is_datetime64_any_dtype(metric_df.get('timestamp')):
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure required columns exist
assert set(['cmdb_id', 'kpi_name', 'value', 'timestamp']).issubset(metric_df.columns), "metric.csv missing required columns"

# 1) Compute global P95 and total_points_full per (cmdb_id, kpi_name) using the entire series
group_cols = ['cmdb_id', 'kpi_name']
p95_series = metric_df.groupby(group_cols, dropna=False)['value'].quantile(0.95)
p95_df = p95_series.reset_index(name='global_p95')

counts_full = metric_df.groupby(group_cols, dropna=False).size().reset_index(name='total_points_full')

meta_df = p95_df.merge(counts_full, on=group_cols, how='left')

# 2) Filter to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Prepare windowed aggregates
# points_in_window and max_value_in_window
points_in_window = window_df.groupby(group_cols, dropna=False).size().reset_index(name='points_in_window')
max_in_window = window_df.groupby(group_cols, dropna=False)['value'].max().reset_index(name='max_value_in_window')

# 4) Compute exceedances by merging global_p95 into windowed rows then checking value > global_p95
if not window_df.empty:
    window_with_p95 = window_df.merge(p95_df, on=group_cols, how='left')
    window_with_p95['exceed'] = window_with_p95['value'] > window_with_p95['global_p95']
    # points_exceeding_p95_in_window
    exceed_counts = (window_with_p95.groupby(group_cols, dropna=False)['exceed']
                     .sum()
                     .reset_index(name='points_exceeding_p95_in_window'))
    # earliest exceedance timestamp per group
    exceed_times = (window_with_p95[window_with_p95['exceed']]
                    .groupby(group_cols, dropna=False)['timestamp']
                    .min()
                    .reset_index(name='earliest_exceedance_timestamp_utc'))
else:
    exceed_counts = pd.DataFrame(columns=group_cols + ['points_exceeding_p95_in_window'])
    exceed_times = pd.DataFrame(columns=group_cols + ['earliest_exceedance_timestamp_utc'])

# 5) Merge all aggregates into final table
result_df = meta_df.merge(points_in_window, on=group_cols, how='left')
result_df = result_df.merge(exceed_counts, on=group_cols, how='left')
result_df = result_df.merge(exceed_times, on=group_cols, how='left')
result_df = result_df.merge(max_in_window, on=group_cols, how='left')

# 6) Fill NaNs for counts with 0, format timestamps and compute ratio
result_df['points_in_window'] = result_df['points_in_window'].fillna(0).astype(int)
result_df['points_exceeding_p95_in_window'] = result_df['points_exceeding_p95_in_window'].fillna(0).astype(int)

# earliest_exceedance: convert to ISO string or blank
def fmt_ts(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(x).isoformat()

if 'earliest_exceedance_timestamp_utc' in result_df.columns:
    result_df['earliest_exceedance_timestamp_utc'] = result_df['earliest_exceedance_timestamp_utc'].apply(fmt_ts)
else:
    result_df['earliest_exceedance_timestamp_utc'] = ""

# max_value_in_window may be NaN for groups with no points in window
# ratio_max_to_p95: set to NaN if global_p95 == 0 or max_value_in_window is NaN
result_df['ratio_max_to_p95'] = np.where(
    (result_df['global_p95'] == 0) | (result_df['max_value_in_window'].isna()),
    np.nan,
    result_df['max_value_in_window'] / result_df['global_p95']
)

# Reorder/select columns as requested
result_df = result_df[[
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'total_points_full',
    'points_in_window',
    'points_exceeding_p95_in_window',
    'earliest_exceedance_timestamp_utc',
    'max_value_in_window',
    'ratio_max_to_p95'
]]

# Sort by points_exceeding_p95_in_window desc, then ratio_max_to_p95 desc
result_df = result_df.sort_values(
    by=['points_exceeding_p95_in_window', 'ratio_max_to_p95'],
    ascending=[False, False],
    na_position='last'
).reset_index(drop=True)

# Limit to top 20 rows
result_df = result_df.head(20)

# Display the summarized table (compact)
result_df

```
Out[3]:
```
summary = (
    "Summary of metric P95 analysis (incident window 2024-01-24 18:55:46 UTC — 2024-01-24 19:25:46 UTC):\n"
    "- Global P95 was computed per (cmdb_id, kpi_name) using the entire available series (not window-limited).\n"
    "- For the top 20 (cmdb_id, kpi_name) results shown: each had 25 total points in the window and 2 points exceeding the global P95.\n"
    "- Top offender by ratio (max_value_in_window / global_p95):\n"
    "  1) ts-assurance-service | cpu — global_p95=0.585076, max_in_window=1.774853, ratio≈3.03, earliest exceedance=2024-01-24T18:58:00+00:00\n"
    "  2) ts-notification-service | cpu — global_p95=0.116200, max_in_window=0.318873, ratio≈2.74, earliest exceedance=2024-01-24T19:13:00+00:00\n"
    "  3) ts-admin-order-service | cpu — global_p95=0.124264, max_in_window=0.317795, ratio≈2.56, earliest exceedance=2024-01-24T19:12:00+00:00\n"
    "- Other top entries are similar (cpu, workload, latency-90, socket KPIs) with ratio between ~1.5 and ~2.4 and earliest exceedances between 18:58 and 19:21 UTC.\n"
    "- Interpretation: multiple services had brief spikes above their global P95 during the incident window; ts-assurance-service (cpu) shows the largest relative spike and should be prioritized for investigation.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  total_points_full  points_in_window  points_exceeding_p95_in_window earliest_exceedance_timestamp_utc  max_value_in_window  ratio_max_to_p95
0        ts-assurance-service         cpu    0.585076                 25                25                               2         2024-01-24T18:58:00+00:00             1.774853          3.033545
1     ts-notification-service         cpu    0.116200                 25                25                               2         2024-01-24T19:13:00+00:00             0.318873          2.744184
2      ts-admin-order-service         cpu    0.124264                 25                25                               2         2024-01-24T19:12:00+00:00             0.317795          2.557426
3           ts-rebook-service         cpu    0.118048                 25                25                               2         2024-01-24T19:03:00+00:00             0.287960          2.439340
4      ts-admin-route-service         cpu    0.129941                 25                25                               2         2024-01-24T19:13:00+00:00             0.311153          2.394567
5      ts-order-other-service         cpu    4.499182                 25                25                               2         2024-01-24T19:12:00+00:00            10.409444          2.313630
6   ts-inside-payment-service         cpu    1.265766                 25                25                               2         2024-01-24T19:02:00+00:00             2.787112          2.201917
7         ts-food-map-service         cpu    1.395147                 25                25                               2         2024-01-24T19:10:00+00:00             2.871594          2.058273
8      ts-order-other-service  latency-90    0.018126                 25                25                               2         2024-01-24T19:21:00+00:00             0.035213          1.942712
9          ts-payment-service    workload    0.148110                 25                25                               2         2024-01-24T19:07:00+00:00             0.280000          1.890487
10   ts-consign-price-service         cpu    0.700433                 25                25                               2         2024-01-24T19:00:00+00:00             1.285965          1.835957
11   ts-ticket-office-service      socket    1.653333                 25                25                               2         2024-01-24T18:58:00+00:00             3.000000          1.814516
12        ts-security-service    workload    0.331441                 25                25                               2         2024-01-24T18:58:00+00:00             0.600286          1.811141
13         ts-payment-service  latency-90    0.035257                 25                25                               2         2024-01-24T19:12:00+00:00             0.057850          1.640819
14        ts-contacts-service    workload    0.632248                 25                25                               2         2024-01-24T18:58:00+00:00             1.028643          1.626961
15   ts-ticket-office-service         cpu    0.015597                 25                25                               2         2024-01-24T18:59:00+00:00             0.025245          1.618605
16            ts-user-service    workload    0.317421                 25                25                               2         2024-01-24T18:58:00+00:00             0.509429          1.604901
17       ts-assurance-service    workload    0.641876                 25                25                               2         2024-01-24T18:58:00+00:00             1.023714          1.594878
18            ts-food-service         cpu    1.263843                 25                25                               2         2024-01-24T19:00:00+00:00             1.948125          1.541429
19  ts-preserve-other-service    workload    0.206229                 25                25                               2         2024-01-24T18:58:00+00:00             0.314429          1.524660```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Load trace.csv if not already in the kernel
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')

# Parse timestamp to UTC if needed
if not pd.api.types.is_datetime64_any_dtype(trace_df.get('timestamp')):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure required columns exist
assert set(['cmdb_id', 'trace_name', 'value', 'timestamp']).issubset(trace_df.columns), "trace.csv missing required columns"

group_cols = ['cmdb_id', 'trace_name']

# 1) Compute global P95 and total_points_full per (cmdb_id, trace_name) using the entire series
p95_series = trace_df.groupby(group_cols, dropna=False)['value'].quantile(0.95)
p95_df = p95_series.reset_index(name='global_p95')

counts_full = trace_df.groupby(group_cols, dropna=False).size().reset_index(name='total_points_full')

meta_df = p95_df.merge(counts_full, on=group_cols, how='left')

# 2) Filter to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Windowed aggregates: points_in_window and max_value_in_window
points_in_window = window_df.groupby(group_cols, dropna=False).size().reset_index(name='points_in_window')
max_in_window = window_df.groupby(group_cols, dropna=False)['value'].max().reset_index(name='max_value_in_window')

# 4) Exceedances: merge global_p95 into window rows, then compute exceed counts and earliest exceedance
if not window_df.empty:
    window_with_p95 = window_df.merge(p95_df, on=group_cols, how='left')
    window_with_p95['exceed'] = window_with_p95['value'] > window_with_p95['global_p95']
    exceed_counts = (window_with_p95.groupby(group_cols, dropna=False)['exceed']
                     .sum()
                     .reset_index(name='points_exceeding_p95_in_window'))
    exceed_times = (window_with_p95[window_with_p95['exceed']]
                    .groupby(group_cols, dropna=False)['timestamp']
                    .min()
                    .reset_index(name='earliest_exceedance_timestamp_utc'))
else:
    exceed_counts = pd.DataFrame(columns=group_cols + ['points_exceeding_p95_in_window'])
    exceed_times = pd.DataFrame(columns=group_cols + ['earliest_exceedance_timestamp_utc'])

# 5) Merge all aggregates into final result
result_df = meta_df.merge(points_in_window, on=group_cols, how='left')
result_df = result_df.merge(exceed_counts, on=group_cols, how='left')
result_df = result_df.merge(exceed_times, on=group_cols, how='left')
result_df = result_df.merge(max_in_window, on=group_cols, how='left')

# 6) Fill and format columns
result_df['points_in_window'] = result_df['points_in_window'].fillna(0).astype(int)
result_df['points_exceeding_p95_in_window'] = result_df['points_exceeding_p95_in_window'].fillna(0).astype(int)

# Format earliest_exceedance_timestamp_utc as ISO string or blank
def fmt_ts(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(x).isoformat()

if 'earliest_exceedance_timestamp_utc' in result_df.columns:
    result_df['earliest_exceedance_timestamp_utc'] = result_df['earliest_exceedance_timestamp_utc'].apply(fmt_ts)
else:
    result_df['earliest_exceedance_timestamp_utc'] = ""

# Compute ratio_max_to_p95, set NaN when global_p95 == 0 or max_value_in_window is NaN
result_df['ratio_max_to_p95'] = np.where(
    (result_df['global_p95'] == 0) | (result_df['max_value_in_window'].isna()),
    np.nan,
    result_df['max_value_in_window'] / result_df['global_p95']
)

# Select and order columns as requested
result_df = result_df[[
    'cmdb_id',
    'trace_name',
    'global_p95',
    'total_points_full',
    'points_in_window',
    'points_exceeding_p95_in_window',
    'earliest_exceedance_timestamp_utc',
    'max_value_in_window',
    'ratio_max_to_p95'
]]

# Sort by points_exceeding_p95_in_window desc, then ratio_max_to_p95 desc
result_df = result_df.sort_values(
    by=['points_exceeding_p95_in_window', 'ratio_max_to_p95'],
    ascending=[False, False],
    na_position='last'
).reset_index(drop=True)

# Limit to top 20 rows
result_df = result_df.head(20)

# Display the summarized table
result_df

```
Out[4]:
```
summary = (
    "Trace P95 analysis summary (global P95 computed over full series; window: 2024-01-24 18:55:46 — 2024-01-24 19:25:46 UTC):\n"
    "- Global P95 was computed per (cmdb_id, trace_name) using the entire available series (not limited to the incident window).\n"
    "- The top 20 trace series all show 23–25 points in the incident window and exactly 2 points exceeding their global P95 during the window.\n"
    "- Largest relative spikes (ratio max/p95):\n"
    "  1) ts-basic-service | trace.to_ts-price-service.duration_p95 — global_p95=0.005857, max=0.034553, ratio≈5.90, earliest exceedance=2024-01-24T19:09:00+00:00\n"
    "  2) ts-price-service | trace.from_ts-basic-service.duration_p95 — same stats and time (symmetrical call)\n"
    "  3) ts-inside-payment-service ↔ ts-payment-service duration_p95 traces — ratio≈5.02, earliest exceedance around 2024-01-24T19:08:00+00:00\n"
    "- Other notable services with >2× spikes: order/travel/preserve/contact flows (ratios ~4.5 → ~2.0) with earliest exceedances roughly between 19:00 and 19:13 UTC.\n"
    "- Timing pattern: many earliest exceedances cluster around 19:08–19:13 UTC (some as early as 19:00 and 19:02), indicating short-lived latency spikes across several service-to-service calls.\n"
    "- Interpretation / recommended next steps: prioritize investigation of ts-basic-service ↔ ts-price-service and the payment-related traces (ts-inside-payment-service ↔ ts-payment-service) because they show the largest relative latency spikes. Then inspect order/travel/preserve/contact flows which also show notable but smaller relative spikes.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  global_p95  total_points_full  points_in_window  points_exceeding_p95_in_window earliest_exceedance_timestamp_utc  max_value_in_window  ratio_max_to_p95
0            ts-basic-service             trace.to_ts-price-service.duration_p95    0.005857                 25                25                               2         2024-01-24T19:09:00+00:00             0.034553          5.899224
1            ts-price-service           trace.from_ts-basic-service.duration_p95    0.005857                 25                25                               2         2024-01-24T19:09:00+00:00             0.034553          5.899224
2   ts-inside-payment-service           trace.to_ts-payment-service.duration_p95    0.014252                 23                23                               2         2024-01-24T19:08:00+00:00             0.071600          5.023724
3          ts-payment-service  trace.from_ts-inside-payment-service.duration_p95    0.014252                 23                23                               2         2024-01-24T19:08:00+00:00             0.071600          5.023724
4      ts-order-other-service         trace.from_ts-travel2-service.duration_p95    0.010713                 25                25                               2         2024-01-24T19:13:00+00:00             0.048591          4.535655
5          ts-travel2-service       trace.to_ts-order-other-service.duration_p95    0.010713                 25                25                               2         2024-01-24T19:13:00+00:00             0.048591          4.535655
6      ts-order-other-service  trace.from_ts-preserve-other-service.duration_p95    0.015852                 22                22                               2         2024-01-24T19:09:00+00:00             0.067728          4.272411
7   ts-preserve-other-service       trace.to_ts-order-other-service.duration_p95    0.015852                 22                22                               2         2024-01-24T19:09:00+00:00             0.067728          4.272411
8         ts-contacts-service  trace.from_ts-preserve-other-service.duration_p95    0.014763                 23                23                               2         2024-01-24T19:00:00+00:00             0.062228          4.215072
9   ts-preserve-other-service          trace.to_ts-contacts-service.duration_p95    0.014763                 23                23                               2         2024-01-24T19:00:00+00:00             0.062228          4.215072
10        ts-contacts-service  trace.from_ts-preserve-other-service.duration_...    0.010090                 23                23                               2         2024-01-24T19:00:00+00:00             0.037583          3.724900
11  ts-preserve-other-service         trace.to_ts-contacts-service.duration_mean    0.010090                 23                23                               2         2024-01-24T19:00:00+00:00             0.037583          3.724900
12       ts-assurance-service       trace.from_ts-assurance-service.duration_p95    0.004046                 23                23                               2         2024-01-24T19:10:00+00:00             0.009275          2.292072
13       ts-assurance-service         trace.to_ts-assurance-service.duration_p95    0.004046                 23                23                               2         2024-01-24T19:10:00+00:00             0.009275          2.292072
14            ts-auth-service            trace.from_ts-auth-service.duration_p95    0.009494                 25                25                               2         2024-01-24T19:03:00+00:00             0.020515          2.160749
15            ts-auth-service              trace.to_ts-auth-service.duration_p95    0.009494                 25                25                               2         2024-01-24T19:03:00+00:00             0.020515          2.160749
16  ts-inside-payment-service          trace.to_ts-payment-service.duration_mean    0.013809                 23                23                               2         2024-01-24T19:08:00+00:00             0.028904          2.093173
17         ts-payment-service  trace.from_ts-inside-payment-service.duration_...    0.013809                 23                23                               2         2024-01-24T19:08:00+00:00             0.028904          2.093173
18           ts-basic-service             trace.to_ts-route-service.duration_p95    0.004589                 25                25                               2         2024-01-24T19:02:00+00:00             0.009283          2.022743
19           ts-route-service           trace.from_ts-basic-service.duration_p95    0.004589                 25                25                               2         2024-01-24T19:02:00+00:00             0.009283          2.022743```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Load metric.csv if not already loaded
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')

# Parse timestamp to UTC datetime if needed
if not pd.api.types.is_datetime64_any_dtype(metric_df.get('timestamp')):
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure required columns exist
assert set(['cmdb_id', 'kpi_name', 'value', 'timestamp']).issubset(metric_df.columns), "metric.csv missing required columns"

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire series (global thresholds)
group_cols = ['cmdb_id', 'kpi_name']
global_p95 = metric_df.groupby(group_cols, dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# 2) Filter metric rows to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Merge global_p95 into the windowed rows
window_with_p95 = window_df.merge(global_p95, on=group_cols, how='left')

# 4) Identify consecutive sequences where value > global_p95 for each group
rows = []
if not window_with_p95.empty:
    # Process per (cmdb_id, kpi_name)
    for (cmdb, kpi), grp in window_with_p95.groupby(group_cols, sort=False):
        g = grp.sort_values('timestamp').reset_index(drop=True)
        # Boolean mask where value > global_p95 (strictly greater)
        mask = g['value'] > g['global_p95']
        if not mask.any():
            continue  # no exceedances for this group in window
        # Create sequence id for runs of equal mask value
        seq_id = (mask != mask.shift(fill_value=False)).cumsum()
        g = g.assign(_mask=mask, _seq_id=seq_id)
        # Keep only True runs and aggregate per run
        exceed_runs = g[g['_mask']].groupby('_seq_id')
        for sid, run in exceed_runs:
            seq_start = run['timestamp'].min()
            seq_end = run['timestamp'].max()
            seq_len = int(run.shape[0])
            if seq_len <= 1:
                continue  # filter out isolated single-point spikes
            max_val = float(run['value'].max())
            p95_val = float(run['global_p95'].iloc[0])
            ratio = np.nan if p95_val == 0 else (max_val / p95_val)
            rows.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'seq_start_timestamp_utc': seq_start.isoformat(),
                'seq_end_timestamp_utc': seq_end.isoformat(),
                'seq_length': seq_len,
                'max_value_in_seq': max_val,
                'global_p95': p95_val,
                'ratio_max_to_p95': ratio
            })

# 5) Build result DataFrame
if rows:
    sequences_df = pd.DataFrame(rows)
    # Sort by seq_length desc then ratio desc
    sequences_df = sequences_df.sort_values(
        by=['seq_length', 'ratio_max_to_p95'],
        ascending=[False, False],
        na_position='last'
    ).reset_index(drop=True)
    # Limit to top 20 rows
    sequences_df = sequences_df.head(20)
else:
    # Empty indicator DataFrame with the required columns
    sequences_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','seq_start_timestamp_utc','seq_end_timestamp_utc',
        'seq_length','max_value_in_seq','global_p95','ratio_max_to_p95'
    ])

# Display the compact table (top sequences or empty table)
sequences_df

```
Out[5]:
```
summary = (
    "Consecutive-exceedance sequences in the incident window (2024-01-24 18:55:46 → 19:25:46 UTC):\n"
    "- Found multiple multi-point sequences where metric value > global P95 (single-point spikes were filtered out).\n"
    "- All reported sequences are length=2 (two consecutive minutes exceeding global P95).\n"
    "- Top sequences by severity (ratio max/p95) — highest relative spikes:\n"
    "  1) ts-notification-service | cpu — 19:13 → 19:14 UTC, max=0.3189, p95=0.1162, ratio≈2.74\n"
    "  2) ts-admin-route-service | cpu — 19:13 → 19:14 UTC, ratio≈2.39\n"
    "  3) ts-inside-payment-service | cpu — 19:02 → 19:03 UTC, ratio≈2.20\n"
    "  4) ts-food-map-service | cpu — 19:10 → 19:11 UTC, ratio≈2.06\n"
    "  5) ts-order-other-service | latency-90 — 19:21 → 19:22 UTC, ratio≈1.94\n"
    "- Other notable 2-minute sequences include payment workload, ticket-office socket, price-service cpu, train-service disk I/O, and several route/order related cpu/latency metrics.\n"
    "- Interpretation: the incident produced short (2-minute) contiguous spikes across several services — prioritize investigation of the top CPU spikes (notification, admin-route, inside-payment, food-map) and the payment/order/price/travel related KPIs showing elevated ratios.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    seq_start_timestamp_utc      seq_end_timestamp_utc  seq_length  max_value_in_seq     global_p95  ratio_max_to_p95
0        ts-notification-service         cpu  2024-01-24T19:13:00+00:00  2024-01-24T19:14:00+00:00           2          0.318873       0.116200          2.744184
1         ts-admin-route-service         cpu  2024-01-24T19:13:00+00:00  2024-01-24T19:14:00+00:00           2          0.311153       0.129941          2.394567
2      ts-inside-payment-service         cpu  2024-01-24T19:02:00+00:00  2024-01-24T19:03:00+00:00           2          2.787112       1.265766          2.201917
3            ts-food-map-service         cpu  2024-01-24T19:10:00+00:00  2024-01-24T19:11:00+00:00           2          2.871594       1.395147          2.058273
4         ts-order-other-service  latency-90  2024-01-24T19:21:00+00:00  2024-01-24T19:22:00+00:00           2          0.035213       0.018126          1.942712
5             ts-payment-service    workload  2024-01-24T19:07:00+00:00  2024-01-24T19:08:00+00:00           2          0.280000       0.148110          1.890487
6       ts-ticket-office-service      socket  2024-01-24T18:58:00+00:00  2024-01-24T18:59:00+00:00           2          3.000000       1.653333          1.814516
7             ts-payment-service  latency-90  2024-01-24T19:12:00+00:00  2024-01-24T19:13:00+00:00           2          0.057850       0.035257          1.640819
8      ts-preserve-other-service    workload  2024-01-24T18:58:00+00:00  2024-01-24T18:59:00+00:00           2          0.314429       0.206229          1.524660
9               ts-price-service         cpu  2024-01-24T19:09:00+00:00  2024-01-24T19:10:00+00:00           2          6.687599       4.554979          1.468195
10              ts-train-service      diskio  2024-01-24T18:58:00+00:00  2024-01-24T18:59:00+00:00           2     175143.421746  123061.837022          1.423215
11         ts-admin-user-service         cpu  2024-01-24T19:13:00+00:00  2024-01-24T19:14:00+00:00           2          0.213514       0.150506          1.418643
12         ts-route-plan-service         cpu  2024-01-24T19:13:00+00:00  2024-01-24T19:14:00+00:00           2          0.163419       0.123242          1.326006
13              ts-order-service  latency-90  2024-01-24T19:00:00+00:00  2024-01-24T19:01:00+00:00           2          0.039156       0.030801          1.271262
14            ts-execute-service         cpu  2024-01-24T19:13:00+00:00  2024-01-24T19:14:00+00:00           2          0.209874       0.167379          1.253887
15  ts-verification-code-service         cpu  2024-01-24T18:59:00+00:00  2024-01-24T19:00:00+00:00           2          0.199532       0.167569          1.190745
16             ts-cancel-service         cpu  2024-01-24T19:13:00+00:00  2024-01-24T19:14:00+00:00           2          0.181851       0.155117          1.172348
17               ts-user-service         cpu  2024-01-24T19:10:00+00:00  2024-01-24T19:11:00+00:00           2          0.922613       0.792096          1.164774
18              ts-price-service  latency-90  2024-01-24T19:09:00+00:00  2024-01-24T19:10:00+00:00           2          0.014134       0.012333          1.146038
19        ts-order-other-service    workload  2024-01-24T18:58:00+00:00  2024-01-24T18:59:00+00:00           2         10.485714       9.502883          1.103424```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse trace_df if available, else load
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')

# Parse timestamp to UTC if not already
if not pd.api.types.is_datetime64_any_dtype(trace_df.get('timestamp')):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Validate required columns
assert set(['cmdb_id', 'trace_name', 'value', 'timestamp']).issubset(trace_df.columns), "trace.csv missing required columns"

# 1) Compute global P95 per (cmdb_id, trace_name) using the entire series
group_cols = ['cmdb_id', 'trace_name']
p95_df = trace_df.groupby(group_cols, dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# 2) Filter to incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Keep only rows where trace_name contains any of the specified service tokens
tokens = [
    'ts-inside-payment-service',
    'ts-payment-service',
    'ts-price-service',
    'ts-basic-service',
    'ts-order-other-service',
    'ts-travel2-service',
    'ts-preserve-other-service',
    'ts-contacts-service',
    'ts-assurance-service',
    'ts-auth-service'
]
pattern = "|".join(tokens)
filtered_window = window_df[window_df['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# 4) If no matching rows, return empty table with requested columns
cols_out = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'points_in_window',
    'points_exceeding_p95_in_window',
    'earliest_exceedance_timestamp_utc',
    'max_value_in_window',
    'ratio_max_to_p95'
]
if filtered_window.empty:
    result_df = pd.DataFrame(columns=cols_out)
else:
    # Aggregates within window
    pts_in_window = filtered_window.groupby(group_cols, dropna=False).size().reset_index(name='points_in_window')
    max_in_window = filtered_window.groupby(group_cols, dropna=False)['value'].max().reset_index(name='max_value_in_window')

    # Merge global_p95 into filtered rows to compute exceedances
    merged = filtered_window.merge(p95_df, on=group_cols, how='left')
    merged['exceed'] = merged['value'] > merged['global_p95']

    exceed_counts = merged.groupby(group_cols, dropna=False)['exceed'].sum().reset_index(name='points_exceeding_p95_in_window')
    earliest_exceed = (merged[merged['exceed']]
                       .groupby(group_cols, dropna=False)['timestamp']
                       .min()
                       .reset_index(name='earliest_exceedance_timestamp_utc'))

    # Compose final table by starting from groups present in window
    result_df = pts_in_window.merge(p95_df, on=group_cols, how='left')
    result_df = result_df.merge(exceed_counts, on=group_cols, how='left')
    result_df = result_df.merge(earliest_exceed, on=group_cols, how='left')
    result_df = result_df.merge(max_in_window, on=group_cols, how='left')

    # Fill and format
    result_df['points_in_window'] = result_df['points_in_window'].fillna(0).astype(int)
    result_df['points_exceeding_p95_in_window'] = result_df['points_exceeding_p95_in_window'].fillna(0).astype(int)
    result_df['earliest_exceedance_timestamp_utc'] = result_df['earliest_exceedance_timestamp_utc'].apply(
        lambda x: "" if pd.isna(x) else pd.to_datetime(x).isoformat()
    )

    result_df['ratio_max_to_p95'] = np.where(
        (result_df['global_p95'] == 0) | (result_df['max_value_in_window'].isna()),
        np.nan,
        result_df['max_value_in_window'] / result_df['global_p95']
    )

    # Select and order columns
    result_df = result_df[cols_out]

    # Sort and limit to top 20
    result_df = result_df.sort_values(
        by=['points_exceeding_p95_in_window', 'ratio_max_to_p95'],
        ascending=[False, False],
        na_position='last'
    ).reset_index(drop=True).head(20)

# Display the summarized table
result_df

```
Out[6]:
```
Summary (trace P95 vs window for selected services; window 2024-01-24 18:55:46 → 19:25:46 UTC):

- What was done: global P95 was computed per (cmdb_id, trace_name) over the entire series (not window-limited). Traces were filtered to the incident window and to the listed services. For each series we counted points in window and how many points exceeded the global P95.

- High-level result: every reported trace series had ~22–25 points in the window and exactly 2 points exceeding its global P95 during the window (i.e., two-minute spikes).

- Largest relative spikes (ratio = max_in_window / global_p95), with earliest exceedance time:
  1) ts-basic-service ↔ ts-price-service (trace.to_ts-price-service.duration_p95 and reciprocal) — ratio ≈ 5.90, earliest 2024-01-24T19:09:00Z
  2) ts-inside-payment-service ↔ ts-payment-service (payment-related duration_p95 traces) — ratio ≈ 5.02, earliest ≈ 2024-01-24T19:08:00Z
  3) ts-order-other-service ↔ ts-travel2-service flows — ratio ≈ 4.54, earliest ≈ 2024-01-24T19:13:00Z
  4) ts-order-other ↔ ts-preserve-other and ts-preserve-other ↔ ts-contacts — ratios ≈ 4.27–4.21, earliest ~19:09 and ~19:00Z
  5) Several other flows (contacts/preserve duration_mean, assurance, auth) show 2–4× spikes with earliest exceedances between ~19:00 and ~19:13Z.

- Pattern & interpretation: spikes are short (two points each) and cluster roughly between 19:00 and 19:13 UTC, with strongest relative latency increases observed for basic↔price and payment-related traces. This suggests transient latency spikes affecting price and payment call paths first, then order/travel/preserve/contact flows.

- Recommendation: prioritize investigation of ts-basic-service ↔ ts-price-service and payment-related traces (ts-inside-payment-service / ts-payment-service), then check order/travel/preserve/contact call paths and assurance/auth traces for propagation or common dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  global_p95  points_in_window  points_exceeding_p95_in_window earliest_exceedance_timestamp_utc  max_value_in_window  ratio_max_to_p95
0            ts-basic-service             trace.to_ts-price-service.duration_p95    0.005857                25                               2         2024-01-24T19:09:00+00:00             0.034553          5.899224
1            ts-price-service           trace.from_ts-basic-service.duration_p95    0.005857                25                               2         2024-01-24T19:09:00+00:00             0.034553          5.899224
2   ts-inside-payment-service           trace.to_ts-payment-service.duration_p95    0.014252                23                               2         2024-01-24T19:08:00+00:00             0.071600          5.023724
3          ts-payment-service  trace.from_ts-inside-payment-service.duration_p95    0.014252                23                               2         2024-01-24T19:08:00+00:00             0.071600          5.023724
4      ts-order-other-service         trace.from_ts-travel2-service.duration_p95    0.010713                25                               2         2024-01-24T19:13:00+00:00             0.048591          4.535655
5          ts-travel2-service       trace.to_ts-order-other-service.duration_p95    0.010713                25                               2         2024-01-24T19:13:00+00:00             0.048591          4.535655
6      ts-order-other-service  trace.from_ts-preserve-other-service.duration_p95    0.015852                22                               2         2024-01-24T19:09:00+00:00             0.067728          4.272411
7   ts-preserve-other-service       trace.to_ts-order-other-service.duration_p95    0.015852                22                               2         2024-01-24T19:09:00+00:00             0.067728          4.272411
8         ts-contacts-service  trace.from_ts-preserve-other-service.duration_p95    0.014763                23                               2         2024-01-24T19:00:00+00:00             0.062228          4.215072
9   ts-preserve-other-service          trace.to_ts-contacts-service.duration_p95    0.014763                23                               2         2024-01-24T19:00:00+00:00             0.062228          4.215072
10        ts-contacts-service  trace.from_ts-preserve-other-service.duration_...    0.010090                23                               2         2024-01-24T19:00:00+00:00             0.037583          3.724900
11  ts-preserve-other-service         trace.to_ts-contacts-service.duration_mean    0.010090                23                               2         2024-01-24T19:00:00+00:00             0.037583          3.724900
12       ts-assurance-service       trace.from_ts-assurance-service.duration_p95    0.004046                23                               2         2024-01-24T19:10:00+00:00             0.009275          2.292072
13       ts-assurance-service         trace.to_ts-assurance-service.duration_p95    0.004046                23                               2         2024-01-24T19:10:00+00:00             0.009275          2.292072
14            ts-auth-service            trace.from_ts-auth-service.duration_p95    0.009494                25                               2         2024-01-24T19:03:00+00:00             0.020515          2.160749
15            ts-auth-service              trace.to_ts-auth-service.duration_p95    0.009494                25                               2         2024-01-24T19:03:00+00:00             0.020515          2.160749
16  ts-inside-payment-service          trace.to_ts-payment-service.duration_mean    0.013809                23                               2         2024-01-24T19:08:00+00:00             0.028904          2.093173
17         ts-payment-service  trace.from_ts-inside-payment-service.duration_...    0.013809                23                               2         2024-01-24T19:08:00+00:00             0.028904          2.093173
18           ts-route-service           trace.from_ts-basic-service.duration_p95    0.004589                25                               2         2024-01-24T19:02:00+00:00             0.009283          2.022743
19     ts-order-other-service  trace.from_ts-preserve-other-service.duration_...    0.015483                22                               2         2024-01-24T19:19:00+00:00             0.030695          1.982527```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import re

# Reuse trace_df if available, else load
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')

# Parse timestamp to UTC if not already
if not pd.api.types.is_datetime64_any_dtype(trace_df.get('timestamp')):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Validate required columns
assert set(['cmdb_id', 'trace_name', 'value', 'timestamp']).issubset(trace_df.columns), "trace.csv missing required columns"

# 1) Compute global P95 per (cmdb_id, trace_name) using the entire series (before any filtering)
group_cols = ['cmdb_id', 'trace_name']
p95_df = trace_df.groupby(group_cols, dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# 2) Filter to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Select traces whose trace_name mentions any of the specified services (as source or target)
tokens = [
    'ts-price-service',
    'ts-payment-service',
    'ts-inside-payment-service',
    'ts-order-other-service',
    'ts-travel2-service',
    'ts-preserve-other-service',
    'ts-contacts-service',
    'ts-assurance-service',
    'ts-auth-service',
    'ts-ticket-office-service'
]
pattern = "|".join(re.escape(t) for t in tokens)
filtered_window = window_df[window_df['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# Prepare output columns
cols_out = [
    'cmdb_id',
    'trace_name',
    'matched_service',
    'points_in_window',
    'points_exceeding_p95_in_window',
    'earliest_exceedance_timestamp_utc',
    'max_value_in_window',
    'global_p95',
    'ratio_max_to_p95'
]

if filtered_window.empty:
    result_df = pd.DataFrame(columns=cols_out)
else:
    # Aggregates within window
    pts_in_window = filtered_window.groupby(group_cols, dropna=False).size().reset_index(name='points_in_window')
    max_in_window = filtered_window.groupby(group_cols, dropna=False)['value'].max().reset_index(name='max_value_in_window')

    # Merge global_p95 into filtered rows to compute exceedances
    merged = filtered_window.merge(p95_df, on=group_cols, how='left')
    merged['exceed'] = merged['value'] > merged['global_p95']

    exceed_counts = merged.groupby(group_cols, dropna=False)['exceed'].sum().reset_index(name='points_exceeding_p95_in_window')
    earliest_exceed = (merged[merged['exceed']]
                       .groupby(group_cols, dropna=False)['timestamp']
                       .min()
                       .reset_index(name='earliest_exceedance_timestamp_utc'))

    # Compose final table starting from groups present in window
    result_df = pts_in_window.merge(p95_df, on=group_cols, how='left')
    result_df = result_df.merge(exceed_counts, on=group_cols, how='left')
    result_df = result_df.merge(earliest_exceed, on=group_cols, how='left')
    result_df = result_df.merge(max_in_window, on=group_cols, how='left')

    # Fill and format
    result_df['points_in_window'] = result_df['points_in_window'].fillna(0).astype(int)
    result_df['points_exceeding_p95_in_window'] = result_df['points_exceeding_p95_in_window'].fillna(0).astype(int)
    result_df['earliest_exceedance_timestamp_utc'] = result_df['earliest_exceedance_timestamp_utc'].apply(
        lambda x: "" if pd.isna(x) else pd.to_datetime(x).isoformat()
    )

    result_df['ratio_max_to_p95'] = np.where(
        (result_df['global_p95'] == 0) | (result_df['max_value_in_window'].isna()),
        np.nan,
        result_df['max_value_in_window'] / result_df['global_p95']
    )

    # matched_service: find first token that appears in trace_name
    def match_token(name):
        for t in tokens:
            if t in name:
                return t
        return ""

    result_df['matched_service'] = result_df['trace_name'].apply(match_token)

    # Select and order requested columns
    result_df = result_df[[
        'cmdb_id',
        'trace_name',
        'matched_service',
        'points_in_window',
        'points_exceeding_p95_in_window',
        'earliest_exceedance_timestamp_utc',
        'max_value_in_window',
        'global_p95',
        'ratio_max_to_p95'
    ]]

    # Sort and limit to top 20
    result_df = result_df.sort_values(
        by=['points_exceeding_p95_in_window', 'ratio_max_to_p95'],
        ascending=[False, False],
        na_position='last'
    ).reset_index(drop=True).head(20)

# Display the summarized table
result_df

```
Out[7]:
```
Summary (trace analysis for selected services; window 2024-01-24 18:55:46 → 19:25:46 UTC):

- What was done: computed global P95 per (cmdb_id, trace_name) over the full series, filtered traces to the incident window, and kept traces mentioning the listed services (price, payment, inside-payment, order-other, travel2, preserve-other, contacts, assurance, auth, ticket-office).

- High-level finding: every reported trace series has ~22–25 samples in the window and exactly 2 points exceeding its global P95 (i.e., two-minute spikes).

- Largest relative spikes (ratio = max_in_window / global_p95), with earliest exceedance:
  1) ts-basic-service ↔ ts-price-service (trace.to_ts-price-service.duration_p95 and reciprocal) — ratio ≈ 5.90, earliest 2024-01-24T19:09:00Z.
  2) ts-inside-payment-service ↔ ts-payment-service (duration_p95 traces) — ratio ≈ 5.02, earliest ≈ 2024-01-24T19:08:00Z.
  3) ts-order-other-service ↔ ts-travel2-service — ratio ≈ 4.54, earliest ≈ 2024-01-24T19:13:00Z.
  4) ts-order-other ↔ ts-preserve-other and ts-preserve-other ↔ ts-contacts — ratios ≈ 4.27–4.21, earliest ~19:09 and ~19:00Z.
  5) ts-assurance-service and ts-auth-service traces show smaller but notable spikes (ratios ≈ 2.29 and ≈ 2.16) with earliest exceedances around 19:03–19:10Z.
  6) Several other matched traces (duration_mean/p95 variants) show 2×–4× transient latency increases.

- Pattern: spikes are short (two consecutive points) and cluster roughly between 19:00 and 19:13 UTC.

- Recommendation: prioritize investigation of ts-basic ↔ ts-price and the payment call path (ts-inside-payment / ts-payment) first (largest relative latency increases). Then inspect order/travel/preserve/contact call paths and assurance/auth traces for propagation or shared dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name            matched_service  points_in_window  points_exceeding_p95_in_window earliest_exceedance_timestamp_utc  max_value_in_window  global_p95  ratio_max_to_p95
0            ts-basic-service             trace.to_ts-price-service.duration_p95           ts-price-service                25                               2         2024-01-24T19:09:00+00:00             0.034553    0.005857          5.899224
1   ts-inside-payment-service           trace.to_ts-payment-service.duration_p95         ts-payment-service                23                               2         2024-01-24T19:08:00+00:00             0.071600    0.014252          5.023724
2          ts-payment-service  trace.from_ts-inside-payment-service.duration_p95  ts-inside-payment-service                23                               2         2024-01-24T19:08:00+00:00             0.071600    0.014252          5.023724
3      ts-order-other-service         trace.from_ts-travel2-service.duration_p95         ts-travel2-service                25                               2         2024-01-24T19:13:00+00:00             0.048591    0.010713          4.535655
4          ts-travel2-service       trace.to_ts-order-other-service.duration_p95     ts-order-other-service                25                               2         2024-01-24T19:13:00+00:00             0.048591    0.010713          4.535655
5      ts-order-other-service  trace.from_ts-preserve-other-service.duration_p95  ts-preserve-other-service                22                               2         2024-01-24T19:09:00+00:00             0.067728    0.015852          4.272411
6   ts-preserve-other-service       trace.to_ts-order-other-service.duration_p95     ts-order-other-service                22                               2         2024-01-24T19:09:00+00:00             0.067728    0.015852          4.272411
7         ts-contacts-service  trace.from_ts-preserve-other-service.duration_p95  ts-preserve-other-service                23                               2         2024-01-24T19:00:00+00:00             0.062228    0.014763          4.215072
8   ts-preserve-other-service          trace.to_ts-contacts-service.duration_p95        ts-contacts-service                23                               2         2024-01-24T19:00:00+00:00             0.062228    0.014763          4.215072
9         ts-contacts-service  trace.from_ts-preserve-other-service.duration_...  ts-preserve-other-service                23                               2         2024-01-24T19:00:00+00:00             0.037583    0.010090          3.724900
10  ts-preserve-other-service         trace.to_ts-contacts-service.duration_mean        ts-contacts-service                23                               2         2024-01-24T19:00:00+00:00             0.037583    0.010090          3.724900
11       ts-assurance-service       trace.from_ts-assurance-service.duration_p95       ts-assurance-service                23                               2         2024-01-24T19:10:00+00:00             0.009275    0.004046          2.292072
12       ts-assurance-service         trace.to_ts-assurance-service.duration_p95       ts-assurance-service                23                               2         2024-01-24T19:10:00+00:00             0.009275    0.004046          2.292072
13            ts-auth-service            trace.from_ts-auth-service.duration_p95            ts-auth-service                25                               2         2024-01-24T19:03:00+00:00             0.020515    0.009494          2.160749
14            ts-auth-service              trace.to_ts-auth-service.duration_p95            ts-auth-service                25                               2         2024-01-24T19:03:00+00:00             0.020515    0.009494          2.160749
15  ts-inside-payment-service          trace.to_ts-payment-service.duration_mean         ts-payment-service                23                               2         2024-01-24T19:08:00+00:00             0.028904    0.013809          2.093173
16         ts-payment-service  trace.from_ts-inside-payment-service.duration_...  ts-inside-payment-service                23                               2         2024-01-24T19:08:00+00:00             0.028904    0.013809          2.093173
17     ts-order-other-service  trace.from_ts-preserve-other-service.duration_...  ts-preserve-other-service                22                               2         2024-01-24T19:19:00+00:00             0.030695    0.015483          1.982527
18  ts-preserve-other-service      trace.to_ts-order-other-service.duration_mean     ts-order-other-service                22                               2         2024-01-24T19:19:00+00:00             0.030695    0.015483          1.982527
19           ts-basic-service            trace.to_ts-price-service.duration_mean           ts-price-service                25                               2         2024-01-24T19:09:00+00:00             0.007662    0.004394          1.743675```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse existing log_df if present, else load
if 'log_df' not in globals():
    log_df = pd.read_csv('log.csv')

# Parse timestamp to UTC if not already
if 'timestamp' in log_df.columns and not pd.api.types.is_datetime64_any_dtype(log_df['timestamp']):
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Validate required columns
assert set(['cmdb_id', 'log_name', 'value', 'timestamp']).issubset(log_df.columns), "log.csv missing required columns"

# Candidate services to keep
candidates = {
    'ts-price-service','ts-basic-service','ts-inside-payment-service','ts-payment-service',
    'ts-order-other-service','ts-travel2-service','ts-preserve-other-service','ts-contacts-service',
    'ts-assurance-service','ts-auth-service','ts-notification-service','ts-admin-route-service',
    'ts-ticket-office-service','ts-order-service'
}

# 1) Compute global P95 per (cmdb_id, log_name) using the entire series (before filtering)
group_cols = ['cmdb_id', 'log_name']
p95_df = (
    log_df.groupby(group_cols, dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Filter to incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# 3) Keep only candidate cmdb_ids
window_df = window_df[window_df['cmdb_id'].isin(candidates)].copy()

# Prepare output structure
cols_out = [
    'cmdb_id',
    'log_name',
    'global_p95',
    'points_in_window',
    'points_exceeding_p95_in_window',
    'earliest_exceedance_timestamp_utc',
    'max_value_in_window',
    'ratio_max_to_p95'
]

if window_df.empty:
    result_df = pd.DataFrame(columns=cols_out)
else:
    # Aggregates within window
    pts_in_window = window_df.groupby(group_cols, dropna=False).size().reset_index(name='points_in_window')
    max_in_window = window_df.groupby(group_cols, dropna=False)['value'].max().reset_index(name='max_value_in_window')

    # Merge global_p95 into window rows to compute exceedances
    merged = window_df.merge(p95_df, on=group_cols, how='left')
    merged['exceed'] = merged['value'] > merged['global_p95']

    exceed_counts = (
        merged.groupby(group_cols, dropna=False)['exceed']
        .sum()
        .reset_index(name='points_exceeding_p95_in_window')
    )
    earliest_exceed = (
        merged[merged['exceed']]
        .groupby(group_cols, dropna=False)['timestamp']
        .min()
        .reset_index(name='earliest_exceedance_timestamp_utc')
    )

    # Compose final result starting from groups present in the window
    result_df = pts_in_window.merge(p95_df, on=group_cols, how='left')
    result_df = result_df.merge(exceed_counts, on=group_cols, how='left')
    result_df = result_df.merge(earliest_exceed, on=group_cols, how='left')
    result_df = result_df.merge(max_in_window, on=group_cols, how='left')

    # Fill and format
    result_df['points_in_window'] = result_df['points_in_window'].fillna(0).astype(int)
    result_df['points_exceeding_p95_in_window'] = result_df['points_exceeding_p95_in_window'].fillna(0).astype(int)

    # Format earliest_exceedance_timestamp_utc as ISO string or blank
    result_df['earliest_exceedance_timestamp_utc'] = result_df['earliest_exceedance_timestamp_utc'].apply(
        lambda x: "" if pd.isna(x) else pd.to_datetime(x).isoformat()
    )

    # Compute ratio, set NaN when global_p95 == 0 or max is NaN
    result_df['ratio_max_to_p95'] = np.where(
        (result_df['global_p95'] == 0) | (result_df['max_value_in_window'].isna()),
        np.nan,
        result_df['max_value_in_window'] / result_df['global_p95']
    )

    # Select and order requested columns
    result_df = result_df[cols_out]

    # Sort by points_exceeding_p95_in_window desc then ratio desc; limit to top 20
    result_df = result_df.sort_values(
        by=['points_exceeding_p95_in_window', 'ratio_max_to_p95'],
        ascending=[False, False],
        na_position='last'
    ).reset_index(drop=True).head(20)

# Display the summarized table (compact)
result_df

```
Out[8]:
```
Summary of log analysis (window 2024-01-24 18:55:46 → 19:25:46 UTC):

- What was done: computed global P95 per (cmdb_id, log_name) over the full series, then examined log rows in the incident window for the listed candidate services.

- Main findings:
  - The predominant signal is log.row_count: many services had small spikes above their global P95 during the window (mostly 2 points exceeding P95).
  - Top log.row_count relative spikes (ratio max/p95) — with earliest exceedance:
    1) ts-order-service | log.row_count — global_p95=439.8, max_in_window=491, ratio≈1.12, earliest exceedance 2024-01-24T19:05:00Z
    2) ts-auth-service | log.row_count — global_p95=348.4, max=372, ratio≈1.07, earliest 19:00Z
    3) ts-price-service | log.row_count — global_p95=286.8, max=302, ratio≈1.05, earliest 19:01Z
    4) ts-basic-service | log.row_count — global_p95=2609.4, max=2716, ratio≈1.04, earliest 19:01Z
    5) ts-travel2-service | log.row_count — global_p95=2400.4, max=2496, ratio≈1.04, earliest 19:00Z
  - A few services (ts-assurance-service, ts-contacts-service) had only 1 point exceeding P95 for log.row_count, with higher ratios (~1.28 and ~1.22) around 19:20Z.
  - log.error_count has global P95 = 0 for all candidate services and shows no exceedances in the window (no error spikes recorded).

- Interpretation:
  - The logs indicate brief increases in request/row counts across many services (small magnitude, ~1–12% above the P95 baseline), concentrated near 19:00–19:05 for several services and a later small spike ~19:20 for assurance/contacts.
  - There is no evidence of elevated error counts during the window.

- Recommendation:
  - Investigate traffic/throughput sources around 19:00–19:05 (e.g., upstream load, batch jobs, or synthetic traffic) because the system experienced a transient increase in row_count across multiple services.
  - Because error_count remained zero, focus initial troubleshooting on capacity/latency or downstream dependencies rather than application errors.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  global_p95  points_in_window  points_exceeding_p95_in_window earliest_exceedance_timestamp_utc  max_value_in_window  ratio_max_to_p95
0            ts-order-service    log.row_count       439.8                25                               2         2024-01-24T19:05:00+00:00                  491          1.116417
1             ts-auth-service    log.row_count       348.4                25                               2         2024-01-24T19:00:00+00:00                  372          1.067738
2            ts-price-service    log.row_count       286.8                25                               2         2024-01-24T19:01:00+00:00                  302          1.052999
3            ts-basic-service    log.row_count      2609.4                25                               2         2024-01-24T19:01:00+00:00                 2716          1.040852
4          ts-travel2-service    log.row_count      2400.4                25                               2         2024-01-24T19:00:00+00:00                 2496          1.039827
5   ts-preserve-other-service    log.row_count       109.4                23                               2         2024-01-24T19:07:00+00:00                  112          1.023766
6   ts-inside-payment-service    log.row_count        19.6                25                               2         2024-01-24T18:59:00+00:00                   20          1.020408
7      ts-order-other-service    log.row_count       486.4                25                               2         2024-01-24T19:00:00+00:00                  491          1.009457
8        ts-assurance-service    log.row_count        14.0                25                               1         2024-01-24T19:20:00+00:00                   18          1.285714
9         ts-contacts-service    log.row_count        54.0                25                               1         2024-01-24T19:20:00+00:00                   66          1.222222
10       ts-assurance-service  log.error_count         0.0                25                               0                                                      0               NaN
11            ts-auth-service  log.error_count         0.0                25                               0                                                      0               NaN
12           ts-basic-service  log.error_count         0.0                25                               0                                                      0               NaN
13        ts-contacts-service  log.error_count         0.0                25                               0                                                      0               NaN
14  ts-inside-payment-service  log.error_count         0.0                25                               0                                                      0               NaN
15     ts-order-other-service  log.error_count         0.0                25                               0                                                      0               NaN
16           ts-order-service  log.error_count         0.0                25                               0                                                      0               NaN
17  ts-preserve-other-service  log.error_count         0.0                23                               0                                                      0               NaN
18           ts-price-service  log.error_count         0.0                25                               0                                                      0               NaN
19         ts-travel2-service  log.error_count         0.0                25                               0                                                      0               NaN```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse metric_df if available, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')

# Parse timestamp to UTC if needed
if 'timestamp' in metric_df.columns and not pd.api.types.is_datetime64_any_dtype(metric_df['timestamp']):
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Validate required columns
assert set(['cmdb_id', 'kpi_name', 'value', 'timestamp']).issubset(metric_df.columns), "metric.csv missing required columns"

# Candidate services to keep
candidates = {'ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service'}

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire series (before filtering)
group_cols = ['cmdb_id', 'kpi_name']
p95_df = metric_df.groupby(group_cols, dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# 2) Filter metric_df to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Keep only candidate cmdb_ids
window_df = window_df[window_df['cmdb_id'].isin(candidates)].copy()

# 4) Merge global_p95 into the windowed rows
window_with_p95 = window_df.merge(p95_df, on=group_cols, how='left')

# 5) Identify consecutive sequences where value > global_p95, per (cmdb_id,kpi_name)
rows = []
if not window_with_p95.empty:
    # iterate groups
    for (cmdb, kpi), grp in window_with_p95.groupby(group_cols, sort=False):
        g = grp.sort_values('timestamp').reset_index(drop=True)
        mask = g['value'] > g['global_p95']
        if not mask.any():
            continue
        # seq id increments when mask changes
        seq_id = (mask != mask.shift(fill_value=False)).cumsum()
        g = g.assign(_mask=mask, _seq_id=seq_id)
        exceed_runs = g[g['_mask']].groupby('_seq_id')
        for sid, run in exceed_runs:
            seq_len = int(run.shape[0])
            if seq_len <= 1:
                continue  # exclude single-point sequences
            seq_start = run['timestamp'].min()
            seq_end = run['timestamp'].max()
            max_val = float(run['value'].max())
            p95_val = float(run['global_p95'].iloc[0])
            ratio = np.nan if p95_val == 0 else (max_val / p95_val)
            rows.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'seq_start_timestamp_utc': seq_start.isoformat(),
                'seq_end_timestamp_utc': seq_end.isoformat(),
                'seq_length': seq_len,
                'max_value_in_seq': max_val,
                'global_p95': p95_val,
                'ratio_max_to_p95': ratio
            })

# 6) Build result DataFrame or empty indicator
cols_out = [
    'cmdb_id','kpi_name','seq_start_timestamp_utc','seq_end_timestamp_utc',
    'seq_length','max_value_in_seq','global_p95','ratio_max_to_p95'
]
if rows:
    sequences_df = pd.DataFrame(rows)
    sequences_df = sequences_df.sort_values(
        by=['seq_length','ratio_max_to_p95'],
        ascending=[False, False],
        na_position='last'
    ).reset_index(drop=True).head(20)
else:
    sequences_df = pd.DataFrame(columns=cols_out)

# Display the summarized table (compact)
sequences_df

```
Out[9]:
```
Summary of consecutive P95 exceedance sequences (window 2024-01-24 18:55:46 → 19:25:46 UTC):

- Method: global P95 was computed per (cmdb_id, kpi_name) over the full series (not window-limited). Then metric rows for the listed services were filtered to the incident window and consecutive runs where value > global P95 were detected. Single-point runs were excluded.

- Overall result: seven multi-point sequences found (all are seq_length = 2, i.e., two consecutive minutes exceeding global P95). No longer sequences detected.

- Top sequences (cmdb_id | kpi_name | start → end UTC | max_value | global_p95 | ratio max/p95):
  1) ts-train-service | diskio — 2024-01-24T18:58 → 18:59 UTC — max=175,143.42, p95=123,061.84, ratio≈1.42
  2) ts-order-service | latency-90 — 2024-01-24T19:00 → 19:01 UTC — max=0.03916, p95=0.03080, ratio≈1.27
  3) ts-order-service | latency-50 — 2024-01-24T19:21 → 19:22 UTC — max=0.00793, p95=0.00767, ratio≈1.03
  4) ts-train-service | socket — 2024-01-24T19:20 → 19:21 UTC — max=26.62, p95=25.82, ratio≈1.03
  5) ts-travel-service | socket — 2024-01-24T19:20 → 19:21 UTC — max=27.27, p95=26.74, ratio≈1.02
  6) ts-auth-service | workload — 2024-01-24T19:00 → 19:01 UTC — max=2.9367, p95=2.9145, ratio≈1.01
  7) ts-train-service | mem — 2024-01-24T19:21 → 19:22 UTC — max=261,280,400, p95=261,162,800, ratio≈1.00045
  8) ts-travel-service | mem — 2024-01-24T19:21 → 19:22 UTC — max=271,244,000, p95=271,123,000, ratio≈1.00045

  (All sequences above are 2-minute runs; listed in approximate severity order.)

- Interpretation and recommendation:
  - The most notable relative spike is ts-train-service disk I/O at ~18:58–18:59 UTC (ratio ~1.42). That is the largest relative deviation and should be prioritized for investigation.
  - Next priority is ts-order-service latency at ~19:00–19:01 UTC (ratio ~1.27).
  - Several small socket and memory increases for ts-train-service and ts-travel-service appear around 19:20–19:22 UTC but are mild (ratios ≈1.00–1.03) — likely transient.
  - All events are short (2 consecutive minutes), suggesting brief transient load/latency/io spikes rather than sustained degradation.

- Actionable next steps: investigate ts-train-service disk I/O at ~18:58 and ts-order-service latency at ~19:00 for root cause (disk contention, IO subsystem, or upstream traffic spike), then review related socket and memory metrics for train/travel services for propagation or correlated events.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    seq_start_timestamp_utc      seq_end_timestamp_utc  seq_length  max_value_in_seq    global_p95  ratio_max_to_p95
0   ts-train-service      diskio  2024-01-24T18:58:00+00:00  2024-01-24T18:59:00+00:00           2      1.751434e+05  1.230618e+05          1.423215
1   ts-order-service  latency-90  2024-01-24T19:00:00+00:00  2024-01-24T19:01:00+00:00           2      3.915626e-02  3.080109e-02          1.271262
2   ts-order-service  latency-50  2024-01-24T19:21:00+00:00  2024-01-24T19:22:00+00:00           2      7.927184e-03  7.669857e-03          1.033551
3   ts-train-service      socket  2024-01-24T19:20:00+00:00  2024-01-24T19:21:00+00:00           2      2.661667e+01  2.582000e+01          1.030855
4  ts-travel-service      socket  2024-01-24T19:20:00+00:00  2024-01-24T19:21:00+00:00           2      2.726667e+01  2.673667e+01          1.019823
5    ts-auth-service    workload  2024-01-24T19:00:00+00:00  2024-01-24T19:01:00+00:00           2      2.936700e+00  2.914513e+00          1.007612
6   ts-train-service         mem  2024-01-24T19:21:00+00:00  2024-01-24T19:22:00+00:00           2      2.612804e+08  2.611628e+08          1.000450
7  ts-travel-service         mem  2024-01-24T19:21:00+00:00  2024-01-24T19:22:00+00:00           2      2.712440e+08  2.711230e+08          1.000446```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import re

# Reuse trace_df if present, else load
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')

# Parse timestamp to UTC if needed
if 'timestamp' in trace_df.columns and not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Validate required columns
assert set(['cmdb_id', 'trace_name', 'value', 'timestamp']).issubset(trace_df.columns), "trace.csv missing required columns"

# 1) Compute global P95 per (cmdb_id, trace_name) using entire series (before any window filtering)
group_cols = ['cmdb_id', 'trace_name']
p95_df = trace_df.groupby(group_cols, dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# 2) Filter to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 18:55:46', utc=True)
end = pd.to_datetime('2024-01-24 19:25:46', utc=True)
window_df = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Keep only series whose trace_name mentions any of these services
tokens = [
    'ts-train-service',
    'ts-order-service',
    'ts-travel-service',
    'ts-auth-service',
    'ts-route-service'
]
pattern = "|".join(re.escape(t) for t in tokens)
filtered_window = window_df[window_df['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# If empty, prepare empty result with requested columns
cols_out = [
    'cmdb_id',
    'trace_name',
    'matched_service',
    'direction',
    'global_p95',
    'points_in_window',
    'points_exceeding_p95_in_window',
    'earliest_exceedance_timestamp_utc',
    'max_value_in_window',
    'ratio_max_to_p95'
]

if filtered_window.empty:
    result_df = pd.DataFrame(columns=cols_out)
else:
    # 4) Aggregates within window
    pts_in_window = filtered_window.groupby(group_cols, dropna=False).size().reset_index(name='points_in_window')
    max_in_window = filtered_window.groupby(group_cols, dropna=False)['value'].max().reset_index(name='max_value_in_window')

    # Merge global_p95 into filtered rows to compute exceedances
    merged = filtered_window.merge(p95_df, on=group_cols, how='left')
    merged['exceed'] = merged['value'] > merged['global_p95']

    exceed_counts = merged.groupby(group_cols, dropna=False)['exceed'].sum().reset_index(name='points_exceeding_p95_in_window')
    earliest_exceed = (merged[merged['exceed']]
                       .groupby(group_cols, dropna=False)['timestamp']
                       .min()
                       .reset_index(name='earliest_exceedance_timestamp_utc'))  # keep as datetime for sorting

    # Build result starting from groups present in window
    result_df = pts_in_window.merge(p95_df, on=group_cols, how='left')
    result_df = result_df.merge(exceed_counts, on=group_cols, how='left')
    result_df = result_df.merge(earliest_exceed, on=group_cols, how='left')
    result_df = result_df.merge(max_in_window, on=group_cols, how='left')

    # Fill numeric counts
    result_df['points_in_window'] = result_df['points_in_window'].fillna(0).astype(int)
    result_df['points_exceeding_p95_in_window'] = result_df['points_exceeding_p95_in_window'].fillna(0).astype(int)

    # Compute ratio
    result_df['ratio_max_to_p95'] = np.where(
        (result_df['global_p95'] == 0) | (result_df['max_value_in_window'].isna()),
        np.nan,
        result_df['max_value_in_window'] / result_df['global_p95']
    )

    # matched_service: first token that appears in trace_name
    def match_token(name):
        for t in tokens:
            if t in name:
                return t
        return ""

    result_df['matched_service'] = result_df['trace_name'].apply(match_token)

    # direction: 'to' if 'to_ts-<service_without_ts->' in trace_name, 'from' if 'from_ts-<...>' else 'other'
    def detect_direction(row):
        name = row['trace_name']
        svc = row['matched_service']
        if not svc:
            return 'other'
        svc_suffix = svc[3:] if svc.startswith('ts-') else svc  # remove leading 'ts-' to match trace naming
        if f"to_ts-{svc_suffix}" in name:
            return 'to'
        if f"from_ts-{svc_suffix}" in name:
            return 'from'
        return 'other'

    result_df['direction'] = result_df.apply(detect_direction, axis=1)

    # Format earliest_exceedance_timestamp_utc as ISO string; keep NaT as blank for output but sort by datetime
    result_df['earliest_sort'] = result_df['earliest_exceedance_timestamp_utc']  # datetime or NaT
    result_df['earliest_exceedance_timestamp_utc'] = result_df['earliest_exceedance_timestamp_utc'].apply(
        lambda x: "" if pd.isna(x) else pd.to_datetime(x).isoformat()
    )

    # Select and order columns
    result_df = result_df[[
        'cmdb_id',
        'trace_name',
        'matched_service',
        'direction',
        'global_p95',
        'points_in_window',
        'points_exceeding_p95_in_window',
        'earliest_sort',
        'earliest_exceedance_timestamp_utc',
        'max_value_in_window',
        'ratio_max_to_p95'
    ]]

    # Sort by earliest_exceedance_timestamp_utc ascending (using earliest_sort), then ratio desc
    result_df = result_df.sort_values(
        by=['earliest_sort', 'ratio_max_to_p95'],
        ascending=[True, False],
        na_position='last'
    ).reset_index(drop=True)

    # Drop the helper earliest_sort column and keep requested columns (with earliest_exceedance_timestamp_utc formatted)
    result_df = result_df.rename(columns={'earliest_exceedance_timestamp_utc': 'earliest_exceedance_timestamp_utc_temp'})
    result_df = result_df[[
        'cmdb_id',
        'trace_name',
        'matched_service',
        'direction',
        'global_p95',
        'points_in_window',
        'points_exceeding_p95_in_window',
        'earliest_sort',  # will use this to place earliest ISO next to it
        'earliest_exceedance_timestamp_utc_temp',
        'max_value_in_window',
        'ratio_max_to_p95'
    ]]
    # Rename for final output consistency
    result_df = result_df.rename(columns={
        'earliest_sort': 'earliest_exceedance_timestamp_utc_sort',
        'earliest_exceedance_timestamp_utc_temp': 'earliest_exceedance_timestamp_utc'
    })
    # Drop the sort helper column before final display but keep ordering already applied
    result_df = result_df.drop(columns=['earliest_exceedance_timestamp_utc_sort'])

    # Limit to top 20
    result_df = result_df.head(20)

# Display the summarized table (compact)
result_df

```
Out[10]:
```
Summary of trace exceedance results for train/order/travel/auth/route (window 2024-01-24 18:55:46 → 19:25:46 UTC):

- What was done: global P95 was computed per (cmdb_id, trace_name) over the full series (not window-limited). Traces were filtered to the incident window and then to series mentioning the five services. For each series we counted points in-window, exceedances of the global P95, earliest exceedance time, max in-window, and relative spike (max / global_p95).

- High-level finding: most matched trace series show brief, short-lived spikes — typically 2 points exceeding P95 (i.e., two-minute spikes) with earliest exceedances clustered at ~18:58–19:00 UTC.

- Notable items (examples from the top results):
  - Several "to_ts-route-service" and "to_ts-travel-service" duration series (e.g., ts-travel2-service → route and ts-food-service → travel) show two-point duration spikes around 18:58 UTC (ratios ~1.01–1.09).
  - Order-related inbound traces (many "to_ts-order-service" entries across multiple callers like ts-preserve-service, root, ts-seat-service, ts-security-service) show two-point spikes around 18:59–19:00 UTC. The largest relative order-related spike in this set: ts-preserve-service → order (duration_mean) ratio ≈ 1.58; ts-seat-service → order (duration_p95) ratio ≈ 1.49.
  - Row_count spikes: ts-auth-service row_count increased (2 points) around 19:00 UTC (ratio ≈ 1.07). ts-train-service and ts-travel2-service row_count also show 2-point increases (ratios ≈ 1.04–1.05).
  - Most ratios are modest (~1.01–1.10), indicating small relative increases, except a few callers into order-service showing higher relative increases (~1.49–1.58).

- Direction pattern: many of the affected traces are "to" the target service (calls into order/route/train), indicating incoming latency or load on those endpoints rather than outbound effects.

- Interpretation and recommendation:
  - The incident exhibits short, correlated two-minute spikes across the order/route/travel call paths starting ~18:58–19:00 UTC. While many spikes are small, caller-to-order traces have the largest relative increases and should be prioritized.
  - Investigate inbound request paths to ts-order-service (check callers like ts-preserve-service, ts-seat-service, ts-security-service) and any upstream load changes around 18:58–19:00 UTC. Also check ts-auth and ts-train row_count/throughput for correlated load.
  - Because spikes are brief, look for transient causes (traffic surge, retries, short network hiccup, or a batch) rather than sustained configuration issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name    matched_service direction   global_p95  points_in_window  points_exceeding_p95_in_window earliest_exceedance_timestamp_utc  max_value_in_window  ratio_max_to_p95
0          ts-travel2-service    trace.to_ts-route-service.duration_p95   ts-route-service        to     0.004117                25                               2         2024-01-24T18:58:00+00:00             0.004473          1.086301
1          ts-travel2-service   trace.to_ts-route-service.duration_mean   ts-route-service        to     0.003396                25                               2         2024-01-24T18:58:00+00:00             0.003437          1.011994
2             ts-food-service  trace.to_ts-travel-service.duration_mean  ts-travel-service        to     0.011864                25                               2         2024-01-24T18:58:00+00:00             0.012004          1.011783
3         ts-preserve-service   trace.to_ts-order-service.duration_mean   ts-order-service        to     0.043755                24                               2         2024-01-24T18:59:00+00:00             0.069045          1.577978
4                        root    trace.to_ts-order-service.duration_p95   ts-order-service        to     0.321919                24                               2         2024-01-24T18:59:00+00:00             0.355001          1.102763
5         ts-preserve-service    trace.to_ts-order-service.duration_p95   ts-order-service        to     0.072159                24                               2         2024-01-24T18:59:00+00:00             0.076090          1.054474
6             ts-seat-service    trace.to_ts-order-service.duration_p95   ts-order-service        to     0.028267                25                               2         2024-01-24T19:00:00+00:00             0.042001          1.485846
7             ts-seat-service   trace.to_ts-order-service.duration_mean   ts-order-service        to     0.007360                25                               2         2024-01-24T19:00:00+00:00             0.008515          1.156990
8         ts-security-service   trace.to_ts-order-service.duration_mean   ts-order-service        to     0.022807                25                               2         2024-01-24T19:00:00+00:00             0.025983          1.139280
9   ts-inside-payment-service       trace.to_ts-order-service.row_count   ts-order-service        to     8.000000                23                               1         2024-01-24T19:00:00+00:00             9.000000          1.125000
10                       root       trace.to_ts-order-service.row_count   ts-order-service        to     9.000000                24                               1         2024-01-24T19:00:00+00:00            10.000000          1.111111
11         ts-station-service     trace.from_ts-order-service.row_count   ts-order-service      from     9.000000                24                               1         2024-01-24T19:00:00+00:00            10.000000          1.111111
12        ts-security-service    trace.to_ts-order-service.duration_p95   ts-order-service        to     0.062184                25                               2         2024-01-24T19:00:00+00:00             0.069085          1.110977
13    ts-admin-travel-service      trace.to_ts-travel-service.row_count  ts-travel-service        to    11.000000                25                               1         2024-01-24T19:00:00+00:00            12.000000          1.090909
14            ts-auth-service      trace.from_ts-auth-service.row_count    ts-auth-service      from   348.400000                25                               2         2024-01-24T19:00:00+00:00           372.000000          1.067738
15            ts-auth-service        trace.to_ts-auth-service.row_count    ts-auth-service        to   348.400000                25                               2         2024-01-24T19:00:00+00:00           372.000000          1.067738
16                       root        trace.to_ts-auth-service.row_count    ts-auth-service        to    87.400000                25                               2         2024-01-24T19:00:00+00:00            92.000000          1.052632
17         ts-travel2-service       trace.to_ts-route-service.row_count   ts-route-service        to   620.600000                25                               2         2024-01-24T19:00:00+00:00           650.000000          1.047374
18           ts-train-service     trace.from_ts-train-service.row_count   ts-train-service      from  2665.600000                25                               2         2024-01-24T19:00:00+00:00          2784.000000          1.044418
19           ts-train-service       trace.to_ts-train-service.row_count   ts-train-service        to  2665.600000                25                               2         2024-01-24T19:00:00+00:00          2784.000000          1.044418```
```